In [ ]:
import pandas as pd

home_df = pd.read_csv(
    r"C:\ai_campus\미니프로젝트_마이데이터\csv\re전처리\home_purchases_전처리최종.csv"
)

print("행, 컬럼 수:", home_df.shape)
display(home_df.head())

for i, col in enumerate(home_df.columns, start=1):
    print(f"{i}. {col}")

In [ ]:
region_df = pd.read_csv(
    r"C:\ai_campus\미니프로젝트_마이데이터\csv\raw 데이터\시,구 분리2차 - 시,구 분리2차.csv"
)

print("행, 컬럼 수:", region_df.shape)
print(region_df.columns.tolist())
display(region_df.head())

In [ ]:
# 1. 고객 ID와 지역 컬럼만 선택하고 동일한 중복 제거
region_cols = ["full_region_name", "sido", "si", "gu", "gun"]

region_map = (
    region_df[["customer_id"] + region_cols]
    .drop_duplicates()
)

# 2. 같은 고객에게 서로 다른 지역 정보가 있는지 확인
assert not region_map["customer_id"].duplicated().any(), \
    "같은 고객 ID에 서로 다른 지역 정보가 있어요."

# 3. 고객 ID 기준으로 지역 추가
home_region_df = home_df.merge(
    region_map,
    on="customer_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

# 4. 행 수와 매칭 결과 확인
assert len(home_region_df) == len(home_df), "병합 후 행 수가 달라졌어요."

print("병합 전:", home_df.shape)
print("병합 후:", home_region_df.drop(columns="_merge").shape)
print("지역 매칭 실패:", home_region_df["_merge"].eq("left_only").sum())

display(
    home_region_df[
        ["customer_id", "apt_name"] + region_cols + ["_merge"]
    ].head()
)

# 모든 고객의 지역이 연결됐을 때만 home_df에 반영
assert home_region_df["_merge"].eq("both").all(), \
    "지역이 연결되지 않은 고객이 있어요."

home_df = home_region_df.drop(columns="_merge")

In [ ]:
print(f"행 수: {home_df.shape[0]:,}")
print(f"컬럼 수: {home_df.shape[1]}")

for i, col in enumerate(home_df.columns, start=1):
    print(f"{i}. {col}")

In [ ]:
region_cols = ["full_region_name", "sido", "si", "gu", "gun"]

# 기존 순서에서 지역 컬럼만 제외
cols = [col for col in home_df.columns if col not in region_cols]

# customer_id 바로 뒤에 지역 컬럼 배치
pos = cols.index("customer_id") + 1
cols[pos:pos] = region_cols

home_df = home_df[cols]

display(home_df.head())

In [ ]:
from pandas.testing import assert_frame_equal

# 기존 최종본 다시 불러오기
original_df = pd.read_csv(
    r"C:\ai_campus\미니프로젝트_마이데이터\csv\re전처리\home_purchases_전처리최종.csv"
)

region_cols = ["full_region_name", "sido", "si", "gu", "gun"]

# 1. 기존 컬럼이 빠지거나 다른 컬럼이 추가되지 않았는지
assert set(home_df.columns) == set(original_df.columns) | set(region_cols)
assert home_df.columns.is_unique, "중복된 컬럼명이 있어요."

# 2. 기존 98개 컬럼의 모든 값과 행 순서 비교
assert_frame_equal(
    original_df.reset_index(drop=True),
    home_df[original_df.columns].reset_index(drop=True),
    check_dtype=False,
    check_exact=True
)

# 3. 추가한 지역 정보가 받은 CSV와 일치하는지
expected_region = (
    region_df[["customer_id"] + region_cols]
    .drop_duplicates()
)

expected = original_df[["customer_id"]].merge(
    expected_region,
    on="customer_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert expected["_merge"].eq("both").all(), "지역 매칭 실패가 있어요."

assert_frame_equal(
    home_df[["customer_id"] + region_cols].reset_index(drop=True),
    expected[["customer_id"] + region_cols].reset_index(drop=True),
    check_dtype=False,
    check_exact=True
)

print("검증 통과: 기존 값과 행 순서가 모두 동일하고, 지역 5개만 정확하게 추가됐어요.")
print(f"{len(home_df):,}행 × {home_df.shape[1]}컬럼")

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal

# 1. 기존 최종본 불러오기
original_df = pd.read_csv(
    r"C:\ai_campus\미니프로젝트_마이데이터\csv\re전처리\home_purchases_전처리최종.csv"
)

region_cols = ["full_region_name", "sido", "si", "gu", "gun"]

# 2. 추가·삭제된 컬럼과 개수 확인
original_cols = original_df.columns.tolist()
current_cols = home_df.columns.tolist()

added_cols = [col for col in current_cols if col not in original_cols]
removed_cols = [col for col in original_cols if col not in current_cols]

expected_count = len(original_cols) - len(removed_cols) + len(added_cols)

print("=== 컬럼 개수 확인 ===")
print(f"기존 컬럼: {len(original_cols)}개")
print(f"삭제 컬럼: {len(removed_cols)}개 → {removed_cols}")
print(f"추가 컬럼: {len(added_cols)}개 → {added_cols}")
print(
    f"계산: {len(original_cols)} - {len(removed_cols)} "
    f"+ {len(added_cols)} = {expected_count}개"
)
print(f"실제 최종 컬럼: {len(current_cols)}개")

assert home_df.columns.is_unique, "중복된 컬럼명이 있어요."
assert len(current_cols) == expected_count, "컬럼 개수 계산이 맞지 않아요."
assert not removed_cols, "기존 컬럼이 삭제됐어요."
assert set(added_cols) == set(region_cols), "지역 5개 외에 컬럼 변화가 있어요."

# 3. 행 수와 고객 ID 중복 확인
print("\n=== 행 수와 고객 ID 확인 ===")
print(f"기존 행 수: {len(original_df):,}행")
print(f"최종 행 수: {len(home_df):,}행")
print(f"고객 ID 중복: {home_df['customer_id'].duplicated().sum():,}건")

assert len(home_df) == len(original_df), "행 수가 달라졌어요."
assert not home_df["customer_id"].duplicated().any(), "고객 ID가 중복돼요."

# 4. 기존 컬럼의 모든 값과 행 순서 비교
assert_frame_equal(
    original_df.reset_index(drop=True),
    home_df[original_cols].reset_index(drop=True),
    check_dtype=False,
    check_exact=True
)

print("\n기존 데이터: 모든 값과 행 순서 동일")

# 5. 받은 CSV의 지역 정보와 비교
region_map = (
    region_df[["customer_id"] + region_cols]
    .drop_duplicates()
)

expected_region = original_df[["customer_id"]].merge(
    region_map,
    on="customer_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

unmatched = expected_region["_merge"].eq("left_only").sum()
print(f"지역 매칭 실패: {unmatched:,}건")

assert unmatched == 0, "지역이 연결되지 않은 고객이 있어요."

assert_frame_equal(
    home_df[["customer_id"] + region_cols].reset_index(drop=True),
    expected_region[["customer_id"] + region_cols].reset_index(drop=True),
    check_dtype=False,
    check_exact=True
)

print("추가 지역 정보: 받은 CSV와 모두 동일")
print("\n검증 통과!")

In [ ]:
home_df.to_csv(
    r"C:\ai_campus\미니프로젝트_마이데이터\csv\re전처리\home_purchases_지역추가.csv",
    index=False,
    encoding="utf-8-sig"
)